In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from modeling.wait_time_forecasting.paths import notebook_context
from modeling.wait_time_forecasting.create_dfs import create_train_df
from modeling.wait_time_forecasting.gradient_boosting_waiting_time import (
    DEFAULT_PARK_NAME,
    load_model_bundle,
)
from modeling.wait_time_forecasting.weather_forecast import (
    build_weather_forecast_csv,
)

from modeling.wait_time_forecasting.waiting_time_pipeline import (
    DEFAULT_WEATHER_FORECAST_PATH,
    DEFAULT_FORECAST_OUTPUT_DIR,
    forecast_for_attractions,
)

ctx = notebook_context(project_root=PROJECT_ROOT, create_models_dir=True)
PROJECT_ROOT = ctx["project_root"]
RAW_DATA_DIR = ctx["data_dir"]
MODELS_DIR = ctx["models_dir"]

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DATA_DIR:", RAW_DATA_DIR)
print("MODELS_DIR:", MODELS_DIR)
print("WEATHER_FORECAST_PATH:", DEFAULT_WEATHER_FORECAST_PATH)
print("FORECAST_OUTPUT_DIR:", DEFAULT_FORECAST_OUTPUT_DIR)


PROJECT_ROOT: C:\Users\artur\Desktop\ESSEC MS IN DATA SCIENCE\0 STUDY\M2\Hackathon\code
RAW_DATA_DIR: C:\Users\artur\Desktop\ESSEC MS IN DATA SCIENCE\0 STUDY\M2\Hackathon\code\modeling\data\raw
MODELS_DIR: C:\Users\artur\Desktop\ESSEC MS IN DATA SCIENCE\0 STUDY\M2\Hackathon\code\modeling\artifacts\models
WEATHER_FORECAST_PATH: C:\Users\artur\Desktop\ESSEC MS IN DATA SCIENCE\0 STUDY\M2\Hackathon\code\modeling\data\processed\weather_forecasted_data.csv
FORECAST_OUTPUT_DIR: C:\Users\artur\Desktop\ESSEC MS IN DATA SCIENCE\0 STUDY\M2\Hackathon\code\modeling\data\forecasts


In [2]:
weather_path = build_weather_forecast_csv()
print("Updated weather forecast:", weather_path)

Updated weather forecast: C:\Users\artur\Desktop\ESSEC MS IN DATA SCIENCE\0 STUDY\M2\Hackathon\code\modeling\data\processed\weather_forecasted_data.csv


In [2]:
train_df = create_train_df(data_dir=RAW_DATA_DIR, park_name=DEFAULT_PARK_NAME)
all_attractions = sorted(train_df["ENTITY_DESCRIPTION_SHORT"].astype(str).str.strip().unique())

print("Park:", DEFAULT_PARK_NAME)
print("Attractions in data:", len(all_attractions))
all_attractions[:20]


Park: PortAventura World
Attractions in data: 26


['Bumper Cars',
 'Bungee Jump',
 'Circus Train',
 'Crazy Dance',
 'Dizzy Dropper',
 'Drop Tower',
 'Flying Coaster',
 'Free Fall',
 'Giant Wheel',
 'Giga Coaster',
 'Go-Karts',
 'Haunted House',
 'Himalaya Ride',
 'Inverted Coaster',
 'Kiddie Coaster',
 'Merry Go Round',
 'Oz Theatre',
 'Rapids Ride',
 'Roller Coaster',
 'Spinning Coaster']

In [3]:
model_bundles = {}
missing_models = []

for attraction_name in all_attractions:
    try:
        bundle, model_path = load_model_bundle(attraction_name=attraction_name)
        model_bundles[attraction_name] = bundle
    except FileNotFoundError:
        missing_models.append(attraction_name)

print("Loaded models:", len(model_bundles))
print("Missing models:", len(missing_models))
if missing_models:
    print("Missing (first 20):", missing_models[:20])

if not model_bundles:
    raise ValueError("No waiting-time models found. Train models first.")


Loaded models: 23
Missing models: 3
Missing (first 20): ['Giga Coaster', 'Himalaya Ride', 'Vertical Drop']


In [4]:
forecast_df = forecast_for_attractions(
    model_bundles=model_bundles,
    data_dir=RAW_DATA_DIR,
    park_name=DEFAULT_PARK_NAME,
    horizon_days=7,
    weather_forecast_path=DEFAULT_WEATHER_FORECAST_PATH,
    output_dir=DEFAULT_FORECAST_OUTPUT_DIR,
)

print("Forecast rows:", len(forecast_df))
print("Forecast start:", pd.to_datetime(forecast_df["date_hour"], errors="coerce").min())
print("Forecast end:", pd.to_datetime(forecast_df["date_hour"], errors="coerce").max())
forecast_df.head()


Skipping Crazy Dance: No weather rows available for forecast window 2022-03-03 21:00:00 to 2022-03-10 20:00:00.
Computed attendance forecast once for 8 day(s).
Saved forecast for Bumper Cars: C:\Users\artur\Desktop\ESSEC MS IN DATA SCIENCE\0 STUDY\M2\Hackathon\code\modeling\data\forecasts\Bumper_Cars_waiting_27_07_2022.csv
Saved forecast for Bungee Jump: C:\Users\artur\Desktop\ESSEC MS IN DATA SCIENCE\0 STUDY\M2\Hackathon\code\modeling\data\forecasts\Bungee_Jump_waiting_27_07_2022.csv
Saved forecast for Circus Train: C:\Users\artur\Desktop\ESSEC MS IN DATA SCIENCE\0 STUDY\M2\Hackathon\code\modeling\data\forecasts\Circus_Train_waiting_26_07_2022.csv
Saved forecast for Dizzy Dropper: C:\Users\artur\Desktop\ESSEC MS IN DATA SCIENCE\0 STUDY\M2\Hackathon\code\modeling\data\forecasts\Dizzy_Dropper_waiting_27_07_2022.csv
Saved forecast for Drop Tower: C:\Users\artur\Desktop\ESSEC MS IN DATA SCIENCE\0 STUDY\M2\Hackathon\code\modeling\data\forecasts\Drop_Tower_waiting_27_07_2022.csv
Saved forec

,date_hour,ENTITY_DESCRIPTION_SHORT,pred_wait_time
0,2022-07-27 09:00:00,Bumper Cars,6.446715
1,2022-07-27 10:00:00,Bumper Cars,8.431226
2,2022-07-27 11:00:00,Bumper Cars,9.787696
3,2022-07-27 12:00:00,Bumper Cars,14.690760
4,2022-07-27 13:00:00,Bumper Cars,15.222479


In [5]:
summary = (
    forecast_df
    .groupby("ENTITY_DESCRIPTION_SHORT", as_index=False)["pred_wait_time"]
    .agg(["count", "mean", "max"])
    .sort_values("mean", ascending=False)
)
summary.head(20)

,ENTITY_DESCRIPTION_SHORT,count,mean,max
17,Spiral Slide,70,81.784243,96.271008
6,Free Fall,70,55.999954,64.684667
19,Swing Ride,98,46.350561,58.372280
7,Giant Wheel,98,44.930018,55.321755
10,Inverted Coaster,70,36.978459,60.678243
21,Zipline,98,31.108993,41.487400
15,Roller Coaster,56,30.365436,36.571863
8,Go-Karts,98,28.353615,41.003027
4,Drop Tower,98,28.140930,42.257986
20,Water Ride,98,23.959583,31.098131


In [6]:
saved_files = sorted(DEFAULT_FORECAST_OUTPUT_DIR.glob("*_waiting_*.csv"))
print("Saved forecast files:", len(saved_files))
saved_files[:20]

Saved forecast files: 23


[WindowsPath('C:/Users/artur/Desktop/ESSEC MS IN DATA SCIENCE/0 STUDY/M2/Hackathon/code/modeling/data/forecasts/Bumper_Cars_waiting_27_07_2022.csv'),
 WindowsPath('C:/Users/artur/Desktop/ESSEC MS IN DATA SCIENCE/0 STUDY/M2/Hackathon/code/modeling/data/forecasts/Bungee_Jump_waiting_27_07_2022.csv'),
 WindowsPath('C:/Users/artur/Desktop/ESSEC MS IN DATA SCIENCE/0 STUDY/M2/Hackathon/code/modeling/data/forecasts/Circus_Train_waiting_26_07_2022.csv'),
 WindowsPath('C:/Users/artur/Desktop/ESSEC MS IN DATA SCIENCE/0 STUDY/M2/Hackathon/code/modeling/data/forecasts/Crazy_Dance_waiting_03_03_2022.csv'),
 WindowsPath('C:/Users/artur/Desktop/ESSEC MS IN DATA SCIENCE/0 STUDY/M2/Hackathon/code/modeling/data/forecasts/Dizzy_Dropper_waiting_27_07_2022.csv'),
 WindowsPath('C:/Users/artur/Desktop/ESSEC MS IN DATA SCIENCE/0 STUDY/M2/Hackathon/code/modeling/data/forecasts/Drop_Tower_waiting_27_07_2022.csv'),
 WindowsPath('C:/Users/artur/Desktop/ESSEC MS IN DATA SCIENCE/0 STUDY/M2/Hackathon/code/modeling/d